# Exploring the data and a run

A short tour that calls the `gems` package's own functions (no copied logic):

1. a selection of **feature bands**,
2. the **known faults** over the **study area**,
3. a chosen run's **fault probability map** next to the known faults.

Set `DATA_DIR` and `RUN_DIR` below, or the `GEMS_DATA_DIR` / `GEMS_RUN_DIR` environment variables. By default the notebook reads `../data` and the newest run in `../runs`.

In [ ]:
%matplotlib inline
import json
import os
from pathlib import Path

from gems.preview import (
    feature_bands_figure,
    known_faults_figure,
    preview_figure,
    read_fault_probability_map,
)
from gems.raster import band_names, read_feature_bands, read_grid, read_labels

In [ ]:
DATA_DIR = Path(os.environ.get("GEMS_DATA_DIR", "../data"))
RUNS_DIR = Path(os.environ.get("GEMS_RUNS_DIR", "../runs"))
RUN_DIR = os.environ.get("GEMS_RUN_DIR")  # None = the newest run in RUNS_DIR

FEATURES = DATA_DIR / "gems-geodawn-numerical-features.tif"
LABELS = DATA_DIR / "existing_faults.tif"
BANDS = ["mag_anom", "rtp", "iso_grav_anom", "iso_grav_anom_slope", "geod_shearrate", "det_elev_slope"]

## Feature bands

All available bands, then maps of a selection (each stretched to its own 2nd???98th percentile inside the study area).

In [ ]:
grid = read_grid(FEATURES)
known_faults, study_area = read_labels(LABELS)
print(band_names(FEATURES))
print(f"Grid {grid.shape}, {grid.res[0]:g} m pixels, {grid.crs}")
print(f"Study area: {study_area.mean():.1%} of the grid; known faults: {known_faults[study_area].mean():.2%} of the study area")

In [ ]:
bands, names = read_feature_bands(FEATURES, BANDS)
feature_bands_figure(bands, names, study_area, grid)

## Known faults over the study area

In [ ]:
known_faults_figure(known_faults, study_area, grid)

## A run's predictions next to the known faults

The same picture the runner saves as `preview.png`, drawn from the run's submission.

In [ ]:
if RUN_DIR is None:
    runs = sorted(p for p in RUNS_DIR.iterdir() if (p / "submission.tif").exists())
    run_dir = runs[-1]
else:
    run_dir = Path(RUN_DIR)
print(f"Run: {run_dir}")
metadata_path = run_dir / "metadata.json"
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    print({k: v for k, v in metadata.items() if k not in ("start_time", "end_time")})

probabilities, _ = read_fault_probability_map(run_dir / "submission.tif")
preview_figure(probabilities, known_faults, study_area, grid, title=run_dir.name)